<a href="https://colab.research.google.com/github/silsrinjoy26-cmd/AutoCorrect-Tool/blob/main/AutoCorrect_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install spellchecker

In [ ]:
pip install torch transformers sentencepiece pyspellchecker


In [ ]:
import torch
import re
import random
from transformers import pipeline, T5Tokenizer, T5ForConditionalGeneration
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from spellchecker import SpellChecker

In [ ]:
# Clear CUDA cache to free up memory
torch.cuda.empty_cache()

# --- 1. AIAutocorrect Class (BERT-based contextual corrector) ---
class AIAutocorrect:
    def __init__(self, model_name="bert-base-uncased"):
        print(f"Loading {model_name} for BERT-based correction...")
        self.corrector = pipeline("fill-mask", model=model_name)
        self.threshold = 0.4  # Minimum confidence to suggest a change

    def suggest_contextual_fix(self, sentence, error_word):
        # Replace the suspected error with the [MASK] token
        masked_sentence = sentence.replace(error_word, self.corrector.tokenizer.mask_token)

        # Get predictions
        predictions = self.corrector(masked_sentence)

        results = []
        for pred in predictions:
            if pred['score'] > self.threshold:
                results.append({
                    "word": pred['token_str'],
                    "confidence": round(pred['score'], 4)
                })

        return results

    def auto_fix(self, sentence, suspected_errors):
        fixed_sentence = sentence
        for error in suspected_errors:
            suggestions = self.suggest_contextual_fix(fixed_sentence, error)
            if suggestions:
                # Take the highest probability prediction
                best_fix = suggestions[0]['word']
                # This simple replace is a known limitation for multiple identical errors
                fixed_sentence = fixed_sentence.replace(error, best_fix)

        return fixed_sentence

# --- 2. ErrorDetector Class (SpellChecker-based) ---
class ErrorDetector:
    def __init__(self):
        self.spell = SpellChecker()

    def detect_errors(self, sentence):
        errors = []
        for match in re.finditer(r'\b[a-zA-Z]+\b', sentence):
            word = match.group(0)
            # Lowercase for consistent spell checking, but store original casing for replacement
            original_word = word

            start_index = match.start()
            end_index = match.end()

            # Check if the word is misspelled by comparing with correction
            if self.spell.correction(original_word.lower()) != original_word.lower():
                errors.append({
                    'word': original_word,
                    'start': start_index,
                    'end': end_index
                })
        return errors

# --- 3. T5-based Model Loading and Functions ---
print("Loading t5-base for grammar/fluency correction...")
model_name = "t5-base" # Reduced from t5-large to t5-base
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Noise Generator (for simulated training)
def Scramble_Text(text, noise_level=0.15):
    chars = "abcdefghijklmnopqrstuvwxyz"
    words = text.split()
    new_words = []
    for word in words:
        if random.random() < noise_level and len(word) > 3:
            idx = random.randint(1, len(word) - 2)
            word_list = list(word)
            if random.random() > 0.5:
                word_list[idx], word_list[idx+1] = word_list[idx+1], word_list[idx] # Swap
            else:
                word_list[idx] = random.choice(chars) # Replace
            new_words.append("".join(word_list))
        else:
            new_words.append(word)
    return " ".join(new_words)

# Training Logic (High-Level Simulation)
def train_on_errors(clean_sentences, epochs=25):
    optimizer = AdamW(model.parameters(), lr=5e-5)
    scaler = GradScaler() # Initialize GradScaler for mixed precision
    model.train()

    print(f"Starting Training for {epochs} Epochs...")
    for epoch in range(epochs):
        total_loss = 0
        for text in clean_sentences:
            noisy_text = "gec: " + Scramble_Text(text)

            # Encode inputs and targets
            input_ids = tokenizer(noisy_text, return_tensors="pt").input_ids.to(device)
            labels = tokenizer(text, return_tensors="pt").input_ids.to(device)

            # Forward pass with autocast for mixed precision
            with autocast():
                outputs = model(input_ids=input_ids, labels=labels)
                loss = outputs.loss

            # Backward pass with scaler
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Average Loss: {total_loss/len(clean_sentences):.4f}")

# T5 Inference Function
def big_ai_autocorrect(input_text):
    input_ids = tokenizer("gec: " + input_text, return_tensors="pt").input_ids.to(device)
    with autocast():
        outputs = model.generate(
            input_ids,
            max_length=128,
            num_beams=5, # Lower beams greater stability
            length_penalty=1.0, # Encourage longer, more complete outputs
            early_stopping=True, # Stop generation when an EOS token is predicted
            no_repeat_ngram_size=2 # Prevent repetitive n-grams
        )
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip the 'gec: ' prefix from the output if it exists (case-insensitive check)
    if decoded_output.startswith("gec: "):
        decoded_output = decoded_output[len("gec: "):]
    elif decoded_output.startswith("Gec: "): # Explicitly handle uppercase G
        decoded_output = decoded_output[len("Gec: "):]
    return decoded_output.strip()

# --- 4. HybridAutocorrect Class ---
class HybridAutocorrect:
    def __init__(self, bert_corrector_instance, t5_corrector_function):
        self.error_detector = ErrorDetector()
        self.bert_corrector = bert_corrector_instance
        self.t5_corrector = t5_corrector_function

    def correct_text(self, sentence):
        print(f"\nOriginal Text: {sentence}")

        # Stage 1: Error Detection
        detected_errors_info = self.error_detector.detect_errors(sentence)
        if detected_errors_info:
            print(f"Detected word-level errors: {[e['word'] for e in detected_errors_info]}")
        else:
            print("No initial word-level errors detected by spellchecker.")

        # Extract just the words for BERT's auto_fix method
        suspected_error_words = [error['word'] for error in detected_errors_info]

        # Stage 2: BERT-based word correction for detected spelling errors
        bert_corrected_sentence = sentence
        if suspected_error_words:
            # Need to pass only the unique suspected words to avoid issues with replace
            unique_suspected_words = list(set(suspected_error_words))
            bert_corrected_sentence = self.bert_corrector.auto_fix(sentence, unique_suspected_words)
            print(f"After BERT-based word correction: {bert_corrected_sentence}")
        else:
            print("No BERT-based word corrections applied (no errors detected by spellchecker).")

        # Stage 3: T5-based overall grammar and fluency correction
        t5_final_corrected_sentence = self.t5_corrector(bert_corrected_sentence)
        print(f"Final correction (T5-based grammar/fluency): {t5_final_corrected_sentence}")

        return t5_final_corrected_sentence

# --- 5. Initial Training (if desired, use small dataset) ---
print("\nPerforming initial training (if necessary) for T5 model...")
clean_data = [
    "The artificial intelligence model is very accurate.",
    "Communication is the key to success in modern business.",
    "This system reduces typing errors significantly.",
    "Please let me know if you have any further questions.",
    "The report must be submitted before the deadline on Friday.",
    "We are looking forward to meeting the new team members.",
    "Technology is changing the way we live and work every day.",
    "The quick brown fox jumps over the lazy dog.",
    "Consistency is more important than perfection in the long run.",
    "Artificial neural networks are inspired by the human brain.",
    "The software update will be available for download tomorrow.",
    "Effective leadership requires empathy and clear vision.",
    "Data science helps companies make better informed decisions.",
    "The weather forecast predicts heavy rain for the weekend.",
    "Please review the attached document and provide your feedback.",
    "A positive attitude can make a huge difference in the workplace.",
    "The project was completed ahead of schedule and under budget.",
    "Sustainable energy is crucial for the future of our planet.",
    "Innovation drives progress in almost every industry.",
    "Learning a new language opens up many cultural opportunities.",
    "The presentation was well-received by the board of directors.",
    "Cybersecurity is a top priority for modern organizations.",
    "Regular exercise and a balanced diet are essential for health.",
    "The customer support team is available twenty-four hours a day.",
    "Collaboration is essential for solving complex global problems."
]
# Only train if model is not already fine-tuned, or if explicit retraining is desired
# For this demonstration, we'll run a few epochs to show the training process
train_on_errors(clean_data, epochs=25) # Reduced epochs for quicker demonstration

# --- 6. Instantiate and Demonstrate HybridAutocorrect with User Input ---
print("\nInitializing Hybrid Autocorrect Tool...")
bert_corrector_instance = AIAutocorrect()
hybrid_corrector = HybridAutocorrect(bert_corrector_instance, big_ai_autocorrect)

print("\nAI-Driven Autocorrect Tool Ready! Type 'quit' to exit.")

while True:
    user_input = input("\nEnter a sentence to correct: ")
    if user_input.lower() == 'quit':
        print("Exiting Autocorrect Tool. Goodbye!")
        break
    if not user_input.strip():
        print("Please enter a sentence.")
        continue

    corrected_text = hybrid_corrector.correct_text(user_input)
    print(f"Input: {user_input}")
    print(f"Corrected: {corrected_text}")
    print("--- End of Correction ---")